In [1]:
# Cell 1: Mount Drive, locate baseline, set CWD, check GPU
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

def find_baseline_dir():
    candidates = [
        "/content/drive/MyDrive/final_project/baseline",
        "/content/drive/MyDrive/final_project/baseline/",
    ]
    for p in candidates:
        if os.path.isdir(p):
            return os.path.abspath(p)

    shared_root = "/content/drive/Shareddrives"
    if os.path.isdir(shared_root):
        for root, dirs, _ in os.walk(shared_root):
            if root.endswith("/final_project") and "baseline" in dirs:
                return os.path.abspath(os.path.join(root, "baseline"))

    raise FileNotFoundError("Could not find final_project/baseline in Drive.")

BASE_DIR = find_baseline_dir()
os.chdir(BASE_DIR)

print("BASE_DIR =", BASE_DIR)
print("CWD =", os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
BASE_DIR = /content/drive/MyDrive/final_project/baseline
CWD = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline
CUDA available: True
GPU: NVIDIA L4


In [2]:
# Cell 2: Install pinned deps
import sys, subprocess

# Upgrade tooling
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "-q"])

# Remove conflicts (ignore if missing)
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y",
                 "transformers", "tokenizers", "huggingface-hub",
                 "pandas", "numpy", "tqdm", "pyyaml"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

pkgs = [
    "pyyaml==6.0.1",
    "tqdm==4.66.2",
    "numpy==1.26.4",
    "pandas==2.2.2",
    "huggingface-hub==0.21.4",
    "tokenizers==0.15.2",
    "transformers==4.38.1",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

import yaml, tqdm, numpy, pandas, transformers, huggingface_hub, tokenizers
print("huggingface_hub:", huggingface_hub.__version__)
print("tokenizers:", tokenizers.__version__)
print("transformers:", transformers.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("Installed OK")

huggingface_hub: 0.21.4
tokenizers: 0.15.2
transformers: 4.38.1
numpy: 1.26.4
pandas: 2.2.2
Installed OK


In [3]:
# Cell 3: Create a tiny dummy mlx_lm package
import os

# quantize_mistral_mlx.py does: from mlx_lm.utils import convert
# On Colab, mlx_lm is not available (it's mac-focused). We stub it.
mlx_root = os.path.join(BASE_DIR, "mlx_lm")
os.makedirs(mlx_root, exist_ok=True)

with open(os.path.join(mlx_root, "__init__.py"), "w") as f:
    f.write("# dummy mlx_lm package for Colab\n")

with open(os.path.join(mlx_root, "utils.py"), "w") as f:
    f.write(
        "def convert(*args, **kwargs):\n"
        "    raise RuntimeError('mlx_lm.convert is not supported on this environment (dummy stub).')\n"
    )

print("Created dummy mlx_lm at:", mlx_root)

Created dummy mlx_lm at: /content/drive/MyDrive/final_project/baseline/mlx_lm


In [4]:
# Cell 4 (NEW): Clone the repo (fast local clone), keep outputs on Drive
import os, subprocess

REPO_URL = "https://github.com/ali-mohmmadi/KGP-CuriousLLM.git"
REPO_DIR = "/content/KGP-CuriousLLM"

if not os.path.isdir(REPO_DIR):
    print("Cloning repository into:", REPO_DIR)
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print("Repo already exists. Pulling latest changes...")
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])

print("Repo ready at:", REPO_DIR)
print("Repo root files:", os.listdir(REPO_DIR)[:10])

Cloning repository into: /content/KGP-CuriousLLM
Repo ready at: /content/KGP-CuriousLLM
Repo root files: ['MDR_embedding_main.py', 'grid_search_mistral_main.py', '.git', 'kg_construct_main.py', 'ft_mistral_main.py', 'README.md', 'configs', 'requirements.txt', 'MDR_main.py', 'images']


In [5]:
# Cell 5 (REPLACE): Add repo + baseline to PYTHONPATH, then import
import sys, os

REPO_DIR = "/content/KGP-CuriousLLM"

# 1) baseline (Drive) first so dummy mlx_lm can be found if needed
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

# 2) repo root so `import KGP...` works
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Smoke test imports (repo-faithful)
from KGP.MDR.tokenizer import load_tokenizer
from KGP.KG.mdr_encoder import Retriever_inf
from KGP.KG.train import run
from KGP.LLMs.Mistral.quantize_mistral_mlx import load_config

print("Imports OK")
print("KGP package loaded from:", REPO_DIR)

Imports OK
KGP package loaded from: /content/KGP-CuriousLLM


In [6]:
# Cell 6: Copy 2WikiMultiHopQA dataset into baseline (repo-style path)
import os, shutil, json

src = "/content/drive/MyDrive/final_project/2wikimultihopqa_dev_2020wiki_1000_converted.json"
dst_dir = os.path.join(BASE_DIR, "DATA", "2WikiMQA")
os.makedirs(dst_dir, exist_ok=True)

dst = os.path.join(dst_dir, "2wikimultihopqa_dev_2020wiki_1000_converted.json")

assert os.path.isfile(src), f"Missing source dataset: {src}"

if not os.path.isfile(dst):
    shutil.copyfile(src, dst)

print("Dataset in baseline:", dst)

# Quick sanity check
data = json.load(open(dst, "r"))
print("num_records =", len(data))
print("keys(example[0]) =", list(data[0].keys()))
print("title_chunks_len(example[0]) =", len(data[0].get("title_chunks", [])))

Dataset in baseline: /content/drive/MyDrive/final_project/baseline/DATA/2WikiMQA/2wikimultihopqa_dev_2020wiki_1000_converted.json
num_records = 1000
keys(example[0]) = ['question', 'answer', 'type', 'titles', 'docs_chunks', 'docs', 'title_chunks', 'supports']
title_chunks_len(example[0]) = 223


In [7]:
# Cell 7: Write embedding config for 2WikiMultiHopQA (separate run_id)
import os, yaml, shutil
import torch

cfg_dir = os.path.join(BASE_DIR, "configs", "mdr_embedding")
os.makedirs(cfg_dir, exist_ok=True)

cfg_2wiki = os.path.join(cfg_dir, "mdr_2wikimultihop_embedding.yml")

# Backup existing config (optional)
bak = cfg_2wiki + ".bak"
if os.path.isfile(cfg_2wiki) and not os.path.isfile(bak):
    shutil.copyfile(cfg_2wiki, bak)
    print("Backed up:", bak)

device = "cuda" if torch.cuda.is_available() else "cpu"

args_dict = {
    "root_dir": ".",
    "dataset": "DATA/2WikiMQA/2wikimultihopqa_dev_2020wiki_1000_converted.json",
    "model": {
        # IMPORTANT: unique run_id so outputs never mix with HotpotQA
        "run_id": "2wikimultihopqa_dev2020wiki_1000_old",
        "base_model": "bert-base-uncased",
        # Will be ensured in next cell (created if missing)
        "from_checkpoint": "mdr_best_model_old_patched.pt",
        "batch_size": 1,
        "max_token_len": 200,
        "device": device,
        "save_every": 250000,
    },
    "emb_checkpoint": "",
}

with open(cfg_2wiki, "w") as f:
    yaml.dump(args_dict, f, sort_keys=False)

print("Wrote config:", cfg_2wiki)
print("Preview:", yaml.safe_load(open(cfg_2wiki)))
print("Expected output dir:", os.path.join(BASE_DIR, "DATA", "KG", "emb", f"emb_{args_dict['model']['run_id']}"))

Wrote config: /content/drive/MyDrive/final_project/baseline/configs/mdr_embedding/mdr_2wikimultihop_embedding.yml
Preview: {'root_dir': '.', 'dataset': 'DATA/2WikiMQA/2wikimultihopqa_dev_2020wiki_1000_converted.json', 'model': {'run_id': '2wikimultihopqa_dev2020wiki_1000_old', 'base_model': 'bert-base-uncased', 'from_checkpoint': 'mdr_best_model_old_patched.pt', 'batch_size': 1, 'max_token_len': 200, 'device': 'cuda', 'save_every': 250000}, 'emb_checkpoint': ''}
Expected output dir: /content/drive/MyDrive/final_project/baseline/DATA/KG/emb/emb_2wikimultihopqa_dev2020wiki_1000_old


In [8]:
# Cell 8: Ensure OLD checkpoint is repo-compatible and update 2Wiki config
import os
import numpy as np
import torch
import yaml
from collections import OrderedDict

cfg_2wiki = os.path.join(BASE_DIR, "configs", "mdr_embedding", "mdr_2wikimultihop_embedding.yml")

old_ckpt_path = os.path.join(BASE_DIR, "mdr_best_model_old.pt")
patched_path  = os.path.join(BASE_DIR, "mdr_best_model_old_patched.pt")

assert os.path.isfile(old_ckpt_path), f"Missing checkpoint: {old_ckpt_path}"

def safe_torch_load(path, map_location="cpu"):
    try:
        import torch.serialization
        torch.serialization.add_safe_globals([np.core.multiarray.scalar, np.dtype])
    except Exception:
        pass
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def looks_like_state_dict(obj):
    if not isinstance(obj, (dict, OrderedDict)) or len(obj) == 0:
        return False
    k0 = next(iter(obj.keys()))
    v0 = obj[k0]
    return isinstance(k0, str) and torch.is_tensor(v0)

def extract_state_dict_any(ckpt_obj):
    if isinstance(ckpt_obj, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in ckpt_obj and isinstance(ckpt_obj[key], (dict, OrderedDict)):
                return ckpt_obj[key]
        if looks_like_state_dict(ckpt_obj):
            return ckpt_obj
    if looks_like_state_dict(ckpt_obj):
        return ckpt_obj
    raise TypeError(f"Unrecognized checkpoint format: {type(ckpt_obj)}")

def normalize_state_dict_keys(sd: dict) -> dict:
    out = {}
    for k, v in sd.items():
        if k.startswith("module."):
            k = k[len("module."):]
        if k.startswith("."):
            k = k[1:]

        if k.startswith("encoder.") or k.startswith("project."):
            out[k] = v
            continue

        if k.startswith(("embeddings.", "encoder.", "pooler.")):
            out["encoder." + k] = v
            continue

        out[k] = v
    return out

if os.path.isfile(patched_path):
    print("Patched checkpoint already exists:", patched_path)
else:
    print("Creating patched checkpoint:", patched_path)
    ckpt_obj = safe_torch_load(old_ckpt_path, map_location="cpu")
    sd_raw   = extract_state_dict_any(ckpt_obj)
    sd_norm  = normalize_state_dict_keys(sd_raw)

    # OLD backbone (same as your hotpot embedding run)
    BASE_MODEL = "bert-base-uncased"
    tok, cfg = load_tokenizer(model_name=BASE_MODEL)
    model = Retriever_inf(cfg, base_model=BASE_MODEL)
    model_sd = model.state_dict()

    # Fill any missing keys from init so strict load works
    patched_sd = {k: sd_norm.get(k, model_sd[k]) for k in model_sd.keys()}

    model.load_state_dict(patched_sd, strict=True)
    print("Strict load verification: OK")

    torch.save({"model_state_dict": patched_sd}, patched_path)
    print("Saved:", patched_path)

# Update ONLY the 2Wiki config to use patched checkpoint
cfg = yaml.safe_load(open(cfg_2wiki, "r"))
cfg["model"]["from_checkpoint"] = os.path.basename(patched_path)
with open(cfg_2wiki, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print("Updated config:", cfg_2wiki, "-> from_checkpoint =", cfg["model"]["from_checkpoint"])

Patched checkpoint already exists: /content/drive/MyDrive/final_project/baseline/mdr_best_model_old_patched.pt
Updated config: /content/drive/MyDrive/final_project/baseline/configs/mdr_embedding/mdr_2wikimultihop_embedding.yml -> from_checkpoint = mdr_best_model_old_patched.pt


In [9]:
# Cell 9: Run embeddings for 2WikiMultiHopQA
import os, json
import numpy as np
import torch

from KGP.MDR.tokenizer import load_tokenizer
from KGP.KG.mdr_encoder import Retriever_inf
from KGP.KG.train import run
from KGP.LLMs.Mistral.quantize_mistral_mlx import load_config

def safe_torch_load(path, map_location="cpu"):
    try:
        import torch.serialization
        torch.serialization.add_safe_globals([np.core.multiarray.scalar, np.dtype])
    except Exception:
        pass
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

cfg_path = "./configs/mdr_embedding/mdr_2wikimultihop_embedding.yml"
args = load_config(cfg_path)

data_path = os.path.join(args["root_dir"], args["dataset"])
raw_documents_data = json.load(open(data_path, "r"))
print("Loaded dataset:", data_path, "records =", len(raw_documents_data))

tokenizer, config = load_tokenizer(model_name=args["model"]["base_model"])
model = Retriever_inf(config, base_model=args["model"]["base_model"])

ckpt_path = os.path.join(args["root_dir"], args["model"]["from_checkpoint"])
model_checkpoint = safe_torch_load(ckpt_path, map_location="cpu")
print("Loading model from checkpoint:", ckpt_path)
model.load_state_dict(model_checkpoint["model_state_dict"])
print("Model loaded successfully.")

run(raw_documents_data, model, tokenizer, args)

Loaded dataset: ./DATA/2WikiMQA/2wikimultihopqa_dev_2020wiki_1000_converted.json records = 1000


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Loading model from checkpoint: ./mdr_best_model_old_patched.pt
Model loaded successfully.
Parsing raw data...


100%|██████████| 1000/1000 [00:00<00:00, 30384.26it/s]


Finished...
Raw data saved...
No checkpoint found. Starting from scratch...


100%|██████████| 133890/133890 [24:29<00:00, 91.10it/s]


133890 batches processed. (Total: 133890)


In [10]:
# Cell 10: Verify saved outputs (2WikiMultiHopQA)
import os
import numpy as np
import yaml

cfg_path = os.path.join(BASE_DIR, "configs", "mdr_embedding", "mdr_2wikimultihop_embedding.yml")
run_id = yaml.safe_load(open(cfg_path, "r"))["model"]["run_id"]

out_dir = os.path.join(BASE_DIR, "DATA", "KG", "emb", f"emb_{run_id}")

passages_path = os.path.join(out_dir, "passages.json")
emb_path      = os.path.join(out_dir, "passage.npy")
cfg_saved     = os.path.join(out_dir, "config.yml")

print("Output folder:", out_dir)
print("passages.json exists:", os.path.isfile(passages_path))
print("passage.npy exists:", os.path.isfile(emb_path))
print("config.yml exists:", os.path.isfile(cfg_saved))

embs = np.load(emb_path)
print("Embeddings shape:", embs.shape)
print("First row (first 5 vals):", embs[0][:5])

Output folder: /content/drive/MyDrive/final_project/baseline/DATA/KG/emb/emb_2wikimultihopqa_dev2020wiki_1000_old
passages.json exists: True
passage.npy exists: True
config.yml exists: True
Embeddings shape: (133890, 768)
First row (first 5 vals): [ 0.07198998  1.0247667  -0.947459   -1.2737803  -0.1430088 ]
